In [ ]:
!rm -rf '/content/DIS_Hughen'

In [ ]:
!git clone https://github.com/NU-Academics/DIS_Hughen.git

In [1]:
import numpy as np
import os
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import  accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

In [2]:
#df = pd.read_csv("undersampled_CIC2019_dataset.csv")
df = pd.read_csv("/content/DIS_Hughen/undersampled_CIC2019_dataset.csv")

In [3]:
df.shape

(4685611, 90)

In [4]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])
X = df.drop(columns=["label"], errors="ignore").select_dtypes(include=[np.number])
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)
y = df["label"]
label_mapping = dict(zip(le.classes_, range(len(le.classes_))))
print(label_mapping)

{'BENIGN': 0, 'DrDoS_DNS': 1, 'DrDoS_LDAP': 2, 'DrDoS_MSSQL': 3, 'DrDoS_NTP': 4, 'DrDoS_NetBIOS': 5, 'DrDoS_SNMP': 6, 'DrDoS_SSDP': 7, 'DrDoS_UDP': 8, 'LDAP': 9, 'MSSQL': 10, 'NetBIOS': 11, 'Portmap': 12, 'Syn': 13, 'TFTP': 14, 'UDP': 15, 'UDP-lag': 16, 'UDPLag': 17, 'WebDDoS': 18}


In [5]:
model_classifier = XGBClassifier(
    n_estimators=300,   
    max_depth=6,
    learning_rate=0.03,
    eval_metric="mlogloss",
    random_state=42,
    tree_method="hist",
)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

In [7]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )
        
    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [9]:
input_dim = X.shape[1]
model = Autoencoder(input_dim=input_dim, latent_dim=20)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

for epoch in range(50):
    optimizer.zero_grad()
    output = model(X_tensor)
    loss = criterion(output, X_tensor)
    loss.backward()
    optimizer.step()

In [10]:
with torch.no_grad():
    Z_train = model.encoder(X_train_tensor).numpy()
    Z_test = model.encoder(X_test_tensor).numpy()

model_classifier.fit(Z_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, objective='multi:softprob', ...)

In [11]:
y_pred = model_classifier.predict(Z_test)
y_prob = model_classifier.predict_proba(Z_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("ROC-AUC:", roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted"))

Accuracy: 0.7630460462500653
Weighted F1: 0.7370615293819677
Macro F1: 0.625859427843666
ROC-AUC: 0.9733469812240148


In [12]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=le.classes_))

               precision    recall  f1-score   support

       BENIGN       0.98      0.99      0.99     10655
    DrDoS_DNS       0.68      0.84      0.75    178752
   DrDoS_LDAP       0.59      0.13      0.21     58000
  DrDoS_MSSQL       0.70      0.32      0.44     30000
    DrDoS_NTP       1.00      1.00      1.00    183522
DrDoS_NetBIOS       0.80      0.79      0.80     13000
   DrDoS_SNMP       0.52      0.52      0.52     83000
   DrDoS_SSDP       0.65      0.06      0.11     21330
    DrDoS_UDP       0.53      0.04      0.08     21000
         LDAP       0.49      0.49      0.49     58000
        MSSQL       0.57      0.86      0.68     38000
      NetBIOS       0.77      0.86      0.81     11011
      Portmap       0.70      0.44      0.54      1122
          Syn       0.99      1.00      0.99      8401
         TFTP       1.00      1.00      1.00    190059
          UDP       0.42      0.97      0.59     30000
      UDP-lag       1.00      0.06      0.12      1000
       UD